In [20]:
import numpy as np

In [21]:
def softmax(x):
    rows, cols = x.shape
    result = np.zeros((rows, cols))
    
    for i in range(rows):
        max_val = np.max(x[i])   
        row = x[i] - max_val
        
        exp_vals = np.exp(row)
        
        sum_exp = np.sum(exp_vals)
        
        result[i] = exp_vals / sum_exp
    
    return result

In [22]:
def self_attention(Q, K, V):
    d = Q.shape[-1]
    
    qkt = np.matmul(Q, K.T) / np.sqrt(d)
    softmaxqkt = softmax(qkt)
    output = np.matmul(softmaxqkt, V)
    
    return output

In [26]:

def flash_attention(Q, K, V, b_size=2):
    N, d = Q.shape
    
    O = np.zeros((N, d))
    m = np.full(N, -np.inf)
    l = np.zeros(N)
    
    for j in range(0, N, b_size):
        K_block = K[j:j+b_size]
        V_block = V[j:j+b_size]
        
        for i in range(N):
            q_i = Q[i]
            
            scores = np.dot(K_block, q_i) / np.sqrt(d)
            m_new = max(m[i], np.max(scores))
        
            correction_factor = np.exp(m[i] - m_new) if m[i] != -np.inf else 0.0
            exp_scores = np.exp(scores - m_new)
            l_new = correction_factor * l[i] + np.sum(exp_scores)
            O[i] = correction_factor * O[i] + np.dot(exp_scores, V_block)
            
            
            m[i] = m_new
            l[i] = l_new
    
   
    for i in range(N):
        O[i] = O[i] / l[i]
    
    return O

In [28]:
def test_case(N, d, B):
    np.random.seed(0)
    
    Q = np.random.randn(N, d)
    K = np.random.randn(N, d)
    V = np.random.randn(N, d)
    
    out1 = self_attention(Q, K, V)
    out2 = flash_attention(Q, K, V, b_size=B)
    
    print(f"\nTest: N={N}, d={d}, B={B}")
    
    print("\nNormal Self-Attention Output:")
    print(out1)
    
    print("\nFlash Attention Output:")
    print(out2)
    

In [29]:
test_case(4, 4, 2)
test_case(6, 4, 2)
test_case(8, 4, 2)
test_case(8, 4, 4)


Test: N=4, d=4, B=2

Normal Self-Attention Output:
[[-0.70811753 -0.9003788  -1.29057301  1.06773569]
 [-0.95848087 -1.44845917 -1.36636925  1.45121241]
 [-0.25175844 -0.4358699  -1.00409108  0.63642757]
 [-0.67161157 -1.05378635 -1.11568195  0.94056785]]

Flash Attention Output:
[[-0.70811753 -0.9003788  -1.29057301  1.06773569]
 [-0.95848087 -1.44845917 -1.36636925  1.45121241]
 [-0.25175844 -0.4358699  -1.00409108  0.63642757]
 [-0.67161157 -1.05378635 -1.11568195  0.94056785]]

Test: N=6, d=4, B=2

Normal Self-Attention Output:
[[-0.78107647 -0.69394194 -0.43693899  0.12079245]
 [-1.34354228 -0.28881136 -0.77886447  0.21760735]
 [-0.38651998 -0.39282209 -0.6489997   0.06829232]
 [-0.78661515 -0.46447673 -0.52366915 -0.09124141]
 [-1.11454636 -0.38966529 -0.66553623 -0.03783732]
 [-0.26601699 -0.04774501 -0.47498844 -0.16737374]]

Flash Attention Output:
[[-0.78107647 -0.69394194 -0.43693899  0.12079245]
 [-1.34354228 -0.28881136 -0.77886447  0.21760735]
 [-0.38651998 -0.39282209 -